# 🧠 APVC – Desafio 2
## Redes neuronais convolucionais para distinguir cães de gatos

### 👥 Grupo G11
- **Bernardo Coelho**, nº 98445  
- **Rafael Alexandre Dias Andorinha**, nº 131000  
- **Nuno Martins**, nº 98863  
- **Pedro Fonte Santa**, nº 105306  

---

📅 **Data de entrega:** 30 de março  

📊 **Objetivo deste script:** O presente script tem como objetivo principal a aplicação de **transferência de conhecimento** através da utilização da rede pré-treinada **VGG16** para a tarefa de **classificação binária de imagens de cães e gatos**, no âmbito do **Desafio 2 da unidade curricular de Aprendizagem Profunda para Visão por Computador (APVC)**.  

---

# 📌 Parte 3:

Numa primeira fase, foi realizado o carregamento e preparação do dataset **"Cats and Dogs"**, disponibilizado via Moodle, esta parte é composta por:

### ✅ Passo 1: Descarregar o Dataset
O primeiro passo consiste em descarregar o dataset **"Cats and Dogs"** disponibilizado no Moodle, que contém 2.000 imagens de treino (1.000 cães e 1.000 gatos) e 1.000 imagens de validação (500 cães e 500 gatos).

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

dataset_path = '../cats_and_dogs/'

# Dataset de treino
train_dataset_vgg = image_dataset_from_directory(
    f"{dataset_path}/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True
)

# Dataset de validação original (1.000 imagens)
val_dataset_vgg = image_dataset_from_directory(
    f"{dataset_path}/validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True
)

# Pré-processamento específico da VGG16
train_dataset_vgg = train_dataset_vgg.map(lambda x, y: (vgg_preprocess(x), y))
val_dataset_vgg = val_dataset_vgg.map(lambda x, y: (vgg_preprocess(x), y))

Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.


### ✅ Passo 2: Dividir o Conjunto de Validação
Obtamos por fazer o mesmo processo feito na CNN. Em que o conjunto de validação que temos (1.000 imagens) deve ser dividido em dois novos conjuntos:
1. **Validação** – 500 imagens (250 cães + 250 gatos)
2. **Teste** – 500 imagens (250 cães + 250 gatos)

In [7]:
# Extrair todas as imagens e labels do val_dataset_vgg
all_images_vgg = []
all_labels_vgg = []

for batch in val_dataset_vgg:
    images, labels = batch
    all_images_vgg.extend(images)
    all_labels_vgg.extend(labels)

# Embaralhar
import random
random.seed(42)
dados = list(zip(all_images_vgg, all_labels_vgg))
random.shuffle(dados)
all_images_vgg, all_labels_vgg = zip(*dados)

# Converter para tensores
all_images_vgg = tf.stack(all_images_vgg)
all_labels_vgg = tf.stack(all_labels_vgg)

# Dividir
val_images_vgg = all_images_vgg[:500]
val_labels_vgg = all_labels_vgg[:500]
test_images_vgg = all_images_vgg[500:]
test_labels_vgg = all_labels_vgg[500:]

# Criar datasets
val_dataset_vgg = tf.data.Dataset.from_tensor_slices((val_images_vgg, val_labels_vgg)).batch(BATCH_SIZE)
test_dataset_vgg = tf.data.Dataset.from_tensor_slices((test_images_vgg, test_labels_vgg)).batch(BATCH_SIZE)

2025-03-25 16:41:56.485936: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### ✅ Passo 3: Verificar a Distribuição das Classes

Confirmamos que temos cerca de 250 cães e 250 gatos em cada conjunto.

In [11]:
def contar_classes(dataset, nome="Conjunto"):
    labels = []
    for _, batch_labels in dataset:
        labels.extend(batch_labels.numpy())
    labels = np.array(labels)
    n_caes = np.sum(labels == 0)
    n_gatos = np.sum(labels == 1)
    print(f"{nome} - Cães: {n_caes}, Gatos: {n_gatos}")

contar_classes(val_dataset_vgg, "Validação")
contar_classes(test_dataset_vgg, "Teste")

2025-03-25 16:42:48.699932: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Validação - Cães: 258, Gatos: 242
Teste - Cães: 242, Gatos: 258


2025-03-25 16:42:49.116402: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


---

### 🚀 2ª Rede para Transferência de Conhecimento: VGG16

Nesta secção vamos aplicar transferência de conhecimento utilizando a rede **VGG16**, pré-treinada no dataset ImageNet.

O objectivo é tirar partido do conhecimento adquirido pela VGG16 em tarefas de classificação geral, adaptando a sua estrutura à tarefa binária de classificação entre cães e gatos.  
Para isso, vamos remover as camadas finais originais da rede e adicionar uma nova "cabeça" personalizada para classificação binária, compatível com o nosso problema.

Tal como na abordagem anterior com a MobileNetV2, iremos:

- Congelar as camadas convolucionais da VGG16 (evitando treinar o modelo todo)
- Adicionar camadas densas personalizadas
- Utilizar imagens com dimensão `224x224`, tal como exigido pela VGG16
- Aplicar o pré-processamento específico (`vgg_preprocess`) exigido por esta arquitetura

---

In [ ]:
# Carregar o modelo VGG16 sem o topo (camadas densas originais)
vgg_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Congelar a base da VGG16 (camadas convolucionais)
vgg_base.trainable = False

# Construir o modelo final com a "cabeça" personalizada
vgg_model = models.Sequential([
    vgg_base,
    layers.Flatten(),                      # VGG não tem GlobalAveragePooling, por isso usamos Flatten
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')  # saída binária: Cão ou Gato
])

### ⚙️ Compilar o modelo

In [ ]:
vgg_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### 🧪 Treinar com callbacks

In [ ]:
BEST_MODEL_CHECKPOINT_VGG = keras.callbacks.ModelCheckpoint(
    filepath="tmp/best_vgg16.weights.h5",
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True
)

EARLY_STOPPING_VGG = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5
)

# Treinar o modelo
history_vgg = vgg_model.fit(
    train_dataset_vgg,
    validation_data=val_dataset_vgg,
    epochs=30,
    callbacks=[BEST_MODEL_CHECKPOINT_VGG, EARLY_STOPPING_VGG],
    verbose=2
)

### 📊 Evolução da Loss e Accuracy Durante o Treino

Visualizamos os resultados guardados no `history` para analisar o desempenho do modelo ao longo das épocas.

In [ ]:
# Carregar os melhores pesos do modelo VGG16
vgg_model.load_weights("tmp/best_vgg16.weights.h5")

# Gráficos de loss e accuracy para VGG16
plt.figure(figsize=(10, 4))

# Acurácia
plt.subplot(1, 2, 1)
plt.plot(history_vgg.history['accuracy'], label='Treino')
plt.plot(history_vgg.history['val_accuracy'], label='Validação')
plt.title('Acurácia - VGG16')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_vgg.history['loss'], label='Treino')
plt.plot(history_vgg.history['val_loss'], label='Validação')
plt.title('Loss - VGG16')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

### 🔀 Matriz de Confusão (Conjunto de Teste)

In [ ]:
# Extrair previsões e labels reais
y_true_vgg = []
y_pred_vgg = []

for images, labels in test_dataset_vgg:
    preds = vgg_model.predict(images)
    y_true_vgg.extend(labels.numpy())
    y_pred_vgg.extend((preds > 0.5).astype(int).flatten())

# Matriz de confusão
cm_vgg = confusion_matrix(y_true_vgg, y_pred_vgg)
disp_vgg = ConfusionMatrixDisplay(confusion_matrix=cm_vgg, display_labels=["Gato", "Cão"])
disp_vgg.plot(cmap=plt.cm.Blues)
plt.title("📉 Matriz de Confusão - VGG16")
plt.show()

### 🖼️ Análise Visual dos Erros

Após a avaliação do modelo VGG16 pela da matriz de confusão, foi possível identificar visualmente as imagens que foram **mal classificadas** no conjunto de teste. 

No total, o modelo cometeu os seguintes erros:

- ❌ **XXX imagens de gatos** foram incorretamente classificadas como cães  
- ❌ **YYY imagens de cães** foram incorretamente classificadas como gatos

Abaixo são apresentadas essas imagens, acompanhadas da previsão feita e da confiança associada a cada decisão.

In [ ]:
# Mostrar imagens mal classificadas pela VGG16 (usando o dataset em memória)
for batch in test_dataset_vgg:
    imagens, labels_reais = batch
    previsoes = vgg_model.predict(imagens)

    for i in range(len(imagens)):
        imagem = imagens[i]
        label_real = labels_reais[i].numpy()
        previsao = previsoes[i][0]
        label_prevista = int(previsao > 0.5)

        if label_prevista != label_real:
            classe_real = "Gato" if label_real == 0 else "Cão"
            classe_prevista = "Gato" if label_prevista == 0 else "Cão"
            confianca = (1 - previsao) * 100 if label_prevista == 0 else previsao * 100

            # Para VGG16, não é necessário reverter a normalização
            imagem_visivel = imagem.numpy().astype("uint8")

            plt.imshow(imagem_visivel)
            plt.axis("off")
            plt.title(f"Erro | Verdadeiro: {classe_real} | Previsto: {classe_prevista} ({confianca:.1f}%)")
            plt.show()

---

## 🧑‍🎓🐕 Classificação de Fotos dos Membros do Grupo

Nesta secção final vamos testar o modelo com **imagens reais** dos animais de estimação dos membros do grupo, tal como foi feito na CNN da primeira parte.
Estas imagens **não fazem parte do dataset original**, o que permite verificar se o modelo generaliza bem para novos dados.

In [ ]:
# Prever imagens reais dos membros do grupo com VGG16
for batch in image_dataset_from_directory(
    f"{dataset_path}/fotos_grupo",
    image_size=(224, 224),
    batch_size=1,
    label_mode='int',
    shuffle=False
):
    imagens_originais, labels_reais = batch

    # Guardar imagem original para mostrar
    imagem_para_mostrar = imagens_originais[0].numpy().astype("uint8")

    # Aplicar o pré-processamento da VGG16 (não alterar a original)
    imagens_processadas = vgg_preprocess(imagens_originais)

    # Fazer previsão com o modelo VGG16
    previsao = vgg_model.predict(imagens_processadas)[0][0]

    # Interpretar resultado
    if previsao < 0.5:
        classe_prevista = "Gato"
        confianca = (1 - previsao) * 100
    else:
        classe_prevista = "Cão"
        confianca = previsao * 100

    classe_real = "Gato" if labels_reais[0].numpy() == 0 else "Cão"

    # Mostrar imagem
    plt.imshow(imagem_para_mostrar)
    plt.axis("off")
    plt.title(f"Verdadeiro: {classe_real} | Previsto: {classe_prevista} ({confianca:.1f}%)")
    plt.show()